# Train a tiny 2-D diffusion model and sample from it

Put u18-p04 to u18-p06 together: train a small noise-prediction network on a 2-D point cloud, then generate new points with DDPM sampling. Everything runs on a CPU in a few seconds.

```python
class EpsMLP(nn.Module):
    def forward(self, x, t):        # x: (B, 2) float32, t: (B,) int64 zero-based steps -> predicted noise (B, 2)
        ...

def train_diffusion(data, betas):
    ...
    return model

def sample(model, n, betas, generator):
    ...
    return x                        # (n, 2)
```

**The data.** `data` is a float32 tensor of shape `(4000, 2)`: points on a ring with a little radial noise, made by

```python
def ring(n, seed, radius=1.0, center=(0.0, 0.0), noise=0.05):
    g = torch.Generator().manual_seed(seed)
    ang = torch.rand(n, generator=g) * 2 * math.pi
    r = radius + noise * torch.randn(n, generator=g)
    return torch.stack([r * torch.cos(ang) + center[0], r * torch.sin(ang) + center[1]], dim=1)
```

**The schedule.** `betas = torch.linspace(1e-4, 0.1, 100, dtype=torch.float64)`, so $T = 100$, `alphas = 1 - betas` and `alpha_bars = torch.cumprod(alphas, dim=0)`, with the zero-based indexing of u18-p04: `alpha_bars[i]` is $\bar\alpha_{i+1}$. Here $\bar\alpha_T \approx 0.0056$, so $x_T$ is close to $N(0, I)$.

**`train_diffusion(data, betas)`** builds a network `model` and trains it with the simple loss of u18-l04 (u18-p05): draw a batch of points $x_0$, steps `t = torch.randint(0, T, (B,))` and noise $\varepsilon \sim N(0, I)$, form $x_t = \sqrt{\bar\alpha}\,x_0 + \sqrt{1 - \bar\alpha}\,\varepsilon$ with `alpha_bars[t]`, and minimize the mean squared error between `model(x_t, t)` and $\varepsilon$. The model must predict the **noise**, and it must take `t` as an input. Return the trained `nn.Module`.

**`sample(model, n, betas, generator)`** runs DDPM ancestral sampling (u18-l05, u18-p06): start from $x_T \sim N(0, I)$ of shape `(n, 2)`, and for `i` from $T - 1$ down to 0 apply the reverse step, adding noise with variance $\tilde\beta$ (or $\beta$) on every step except the last. Draw all noise from `generator` (`torch.randn((n, 2), generator=generator)`). Return a tensor of shape `(n, 2)` that does not require grad.

**How it is graded.** The grader trains your model once on the ring above (`ring(4000, seed=1)`), draws 5000 points with your `sample`, and compares them with 5000 fresh points from the same ring. Both sets are binned on a 12 by 12 grid over the square $[-1.5, 1.5]^2$, with one extra bin for points outside it; the **total variation distance** between the two histograms, $\frac12\sum_\text{bins}|p_\text{bin} - q_\text{bin}|$, is the fraction of probability mass in the wrong place. It must be **at most 0.20**. Two fresh sets of ring points are about 0.06 apart; plain $N(0, I)$ noise is 0.74 away. The grader also

- samples your model with its own DDPM sampler (the same target), which catches a model that does not predict the noise;
- checks that your model's output changes with `t`;
- checks that `sample` is reproducible from the generator and returns no graph;
- trains again on a hidden ring with another radius and center (same target).

**Example.** A solution built as in the suggested model below reaches a distance of 0.062 on the example ring and 0.098 on the hidden one, and trains in about 3 seconds on a laptop CPU with 2 threads.

**Suggested model.** Concatenate $x$ with a sinusoidal embedding of $t$ (u18-l04): for 8 frequencies $f_k = 1000^{-k/8}$, $k = 0, \dots, 7$, use $\sin(f_k t)$ and $\cos(f_k t)$, 16 numbers. Feed the 18 inputs to an MLP with three hidden layers of width 128 and SiLU activations, output 2 numbers. Train 3000 steps with batch 256 and Adam at learning rate $2 \times 10^{-3}$, decayed with `torch.optim.lr_scheduler.CosineAnnealingLR`. Set `torch.manual_seed(0)` inside `train_diffusion`. Simpler inputs (appending $t/T$ as one extra number) also pass.

**Restrictions.** Only `import torch` (anything inside it) and `import math`.

**Constraints.** Time limit 40 seconds per call to `train_diffusion` on Colab's CPU. The grader runs on a faster CPU and allows 20 seconds there.

**The full-size version (not graded).** The same loss and sampler train an image model. In Colab choose Runtime, Change runtime type, L4 GPU; set `device = "cuda"` and move the model (`model.to(device)`) and every batch to it. Load MNIST with `torchvision.datasets.MNIST` and scale pixels to $[-1, 1]$. Use the DDPM schedule ($T = 1000$, betas from $10^{-4}$ to $0.02$), a small U-Net that adds the time embedding in every residual block (u18-l04) with 64, 128 and 256 channels at the three resolutions, batch 128, and Adam at learning rate $2 \times 10^{-4}$ for about 20 epochs. Sampling then takes 1000 network calls per batch; the DDIM sampler of u18-l05 with 50 steps cuts that to 50.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem7_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem7_submission.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class EpsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Your code

    def forward(self, x, t):
        # Your code: x (B, 2) float, t (B,) long -> predicted noise (B, 2)
        pass


def train_diffusion(data, betas):
    torch.manual_seed(0)
    # Your code
    return model


@torch.no_grad()
def sample(model, n, betas, generator):
    # Your code
    return x


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem7_submission as submission
importlib.reload(submission)
# e.g. submission.train_diffusion(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XI1z2zaW/1dw7kxNZmVWcj7acercOo3bzU2+JvHO7o3lYyERklhTIEtQsVyv//f7vQeABCU5idO9y04tEgQeHt73ewD2Zi/P9o7E3mr0w0E1/H5vIPaWqpFoutn+0uRNoajtrJa5FlI0ub4WhwcvRJbPZiuTl1osy0wVQupMGLmsCiVmdbkUeUPjC2VMqT3QYviYp8v1qlEGrYeP8UqQ8umqaK7R8ggNSjc1Pe81NGnazsRjy2xVqFTLJaNV1eWkUMvvU7OaLHPje8miKK9UlubLqqwbmul8rynr6YJByGaxd4GHidRa0YrP3VtalGVFvWeyMIqWr0yTNvlSlasmpQ+jwyE1y7mFOVda5kkPQduky9yotKpVlk+b3pcsq5YpEyrX866ZJknVcqKyzLUzvgmTAC2MGmNtylU95cWXdT7PtSyodyXr5hCNTb1St3if5QVT+GZvIWsNLiQVk3TM//ubKipVG9EsZCMcDcW8lhk1WqKJ6FfmoxvuWx/8GidjPdYnrrvIjZiWugHPvlvpvDHf/Ug/z75zUPFe5dmz72xvIJGIl43I1CwHVEH0FbOVZhoZQVzNuDF9YHFr5KXCkxrrjsHCyoD4FU2/JuL0nyev3706/SCK3DS0JGXBEjAjzKK80gKD3q+0iD7mJgdW8VPxgcA1ol5hWvVR1dc8CEv7WebFqsZIEN4oQa/iKm8WkH3AM3KuxFWdN1iwmJW1eK70EeT+WlwRulfS4Gup5wOhCarIVstKLPIsQ/dcV6vGEPUsF7Qj6bSsrtsXkgOir3vVGH8tAFVX1PqN+Kks5ET89O7vRshaCUNyXhOtNKGyb8TrQ/GuLgfClKIop7IgiJAfI5YrEGWiiKggUUk6l+NzkRMdJlj/AqycySnkLqGZDpMhMVeK+Yr4H9mZDWAV6qBZ1EpC4SsFhqFXXa7mi+Ia0lLMYAoIi2lZE6FrRURvRD4jBPHKGunW4VSLiPLh3enpi/Tnk5/O3r4XxzQ7LXisp4XE7MSG6HQ9VRVJSnw01gL/KnyyvSBQoLjKIshiNhBLM/d9MK8uicg6cy30r2NuRH07IKmuorUfCzMUjPEcIr0MWmdYf65NI/VUReuB/Z6cKW3KOg6G87SqWdVarJMMFne6iOJkWq3wl9kcxbaz4mWKlzzdaV2XdQDFLjmApatEGlnX8hp4d+swC1mpaF42Awilpr+Qz+PxHiQblna85zEzoHWzgtWOaOXoHyc81OHCNEUf34lgxQMxG+/dEMBbMNzYucSNgd1R60pNG8jETdD/FtN1mE2L0nwKs4Gom7I4HqmDJwMh3eMPHl9o1hVQbpGVprnGQmdFKQkxauc5+x+Cxczt+mhJV/bxjuXYBbju8a61+U+8vHYCMCQ3MG95g9HAoyiiWMBSkBSG367ct3B6sqSw92QJNY3J9awF7gSZ+F0UjogghiMX/XHkoj+h4Km6BsWYEITcUq7pR05MNBcH4iqOY4KNheZ/KKHg+MSQlG+HqvTpxIZOfJQF7IOIClnPyewqElhxg5+j5OH8Nu7xXv2+ksUXSGVfqWxXQlpnLOmDmCkKJGTT1O77eI/Z0QHZIS6BgHR9/s/FIhQNxj91dAByW8oUUrWFQFw52sQYwkDY8uI7GDDGN/j0H3UPNeqFph4zcpOyijA76MncyQ+eMOBIN2QLf2oFKRzgOEk5VkvTHjpzmD/inJt133fa91D7mKZGywqevPn/Nc1FqdU9jTKm9FK5JpFk374Xc3RM2tvHJ6LocMCBSxzi019cDzsCGGVKVcdniPQCAXMInoF8m+htA4h7LoSaEoLJ39Y9woMtkRyIydfTXX4R3W0Xqxc839eSXRLZGY7pmwI3kbSTmGgS77I1GN7ZmZ3DWzsxcY/M275m0woG1tpBsnXIKw+HIQTBS5kuV1AKRDfRTA/EA5hTaMaDB5dX9OQxsaHjT/AAYqZ55hkFqLQGxDiIAudQL4kUDYNWSwTlcCGiKuRUJT7qZK9P+gS7eN5plow5npU0gGa8sD0RnZHz0NEmQvYzDckhyFiwiWmoQiijagn/9kdeRXYMz9ETcLYnrXA5I3KNxAaymCkK8PNZTpElhczdWr65ycVfxOj2qbgq60uK6iULryD2IR5t7aUjMpDvKExBfK4joygUBFZfSuWnIYlhXcOQe0ckzakn/KFB7I0VEIE41t1vRyJ4tzkUTfUWIGpKwZAgLQ0tiePw13IqppiAhmCRNuqmbIUDewoOkF0cTFYZLCk1cwyOsDynBGpVQZ1Ec5VPFQEEhXJ46HLmNBg4e4oaFiFqmSFeIvdNEDBihW+JOKMPeY1mLKbYpynIL9jJiMIq64lVM6RoEhASZJezlMGo2lubz4hS1uwejBilceGIW++xcEwU34kwbehEMiVRrEkbosNQ7sBETPPjsYO0YYomyGkuuyZ5OBCXh5itbx8Z5cFGY28lnycF40mEOGQyHAbNTAZkbFFGLvYOcoTxZrgg1qOmLFlukPxnzVFyOLtFlA8pU09d1tUKMQsT1I4BiZWmfP7GkcaN4xwSiZ1n8Y5/PCUyS6TBM4jWnAJYm2TeOD4xINKOTyporefQTpV5hnUJDjiZlUtQeyYhfmnbccx1lZWRMi9TrjD0qhy/UAs0AJh1xQOuYk0XeZGR1k2pMoKEOSddoJ7/9eHtG+CqWXNMkwFD1tXqulmg5eCl8POIH12JI83y+pn4sZsDH/EOrfwXtzXPtlJ+KFuQ/oOx66bIJ22T/UFLsmryomsu28ffTNkVCSrZLMLhJp9r2Q0z12ajvOBfaviFiZxeti2QYio2GVtrMNMcy9SqNXdouMybg0KhmxglT5KRi+nMEdWJllTtG7BJQXbhnI83fpXMQeX5WPs5klleQKL9azTeA9plrSgDdJUWRKLJg7/rS001HFMWVFEpuQKACfOKuJY8cJHi307fn0JzHC2Sd/iN0pTKYGkaJzBcNBxRXSVrYDbWoEpCnRNIq6qbaAhlQPhAYGIA3FX74lqQ+Aa++nd5JE4fDQ+5EvbqHyf//SF9fvLmzekLYHAz3ktd4TFNaTFYmOenfVVrNXVPiPPt07RcVsB1vHc71q/evn2Xvnn74vQDwEWQleTnErpKD/+A4Cr7eGKu9bT98Aox5E+AYd8+qODlBYLM7u0Xpck/l/XpuorDEsuZrcV8rsqSSmR6y4hkbLWEvYHbUq3GcqroAYXVCIpspul0oaaXJjL1lFjcyL6/fW/1naJhOCuiY3lpibM0cxDmKcwiRYC2xsdqY5r9ljscDLlSIjmqpralV9NzUdYJnV/simabWiGwYypBSpBfA9F+HPrhGvn5muNQKsqp7fjw3KF9BGnhGrLD/ojMpOHhLkUutbUzNyqhX13eHtEzeiP5uXUIukI2uQTQK4Fhhp5sVLeJRucXDtGJbNvZTzbe5XA5NF0ZBha0T6APqa2VunYScYSM0N08g6pQJFbbQMECBgTbP6KMhoLufEL1fCznUtndgIQE203ANfWA5kvZLFdF2DLj6ghMNdlmEP9KFpcRMWPDdQdhOnW38mxTg82sog1lqWPC6B5tOzBXUkG8HEnuREH5kkoL9jXBbCAkRY7REpEnzBElcwR8ychaRsRHu31jwItEZpmbI97u3PEm6JeYqsiJ3TTn+cHoYsfAjnnJqsoo6t4xNkwSi0+T8WdYvc3VgKHgFdPRFd6x/E6hdtCSRzhC4vlPUHGTgoC2QYWAdD0K7JKAjaF3Ee+rhvI670v0E687m2vfEAjGgxTti6C+CSzypwDm2d2r2t1p14yds9qckxU/kVWldBb5io8dNN6zJs+a9//s02vXLOwFn+f6rfNiJ6v5iSEfFNvaysaApHT9XsvmNULGLZm2NuieyAGxQCADkBQkOlB3mn8fBVgtsSJPrawQ1C/5raT0lA1sFMwTx7uty1/Io4AwHtTN/kDsWyBeo0gB9zEf8t75PpVib3fXE3fhTyWLHegbj3sH7Jt2B3EitSHaThfOQ3Ao2DkILj8riST2n/wbP7XZaHElr82B3Y5FJoO4MG8BkVA6aKUurgd+TmQunauJrBTYIpILpSjouBaFmiM1v6KwSXDPJEniwJkhKBNZSUyxgBc555eWC1HgNL9l5xh4YYuuc74Qw3+JKPCl34peXBiIEGa4t+gAAyMcfQh+ZOIdsgPI8Z/hsC79HO2WbMdlYN5ffGqVKC0rG0/2q23U3SrZhup9yYIpo4Rc/FV42JSyBnojIhs5uRnOhxehcO9Y/n1I0M36qcWzcdtaMrd+3YJtmMTC3lsomkmNSahrtVCak9g2duRR/971u5XdfiJTn9U5wBTXKUeyEUAg0U0pm+pH9G+1OmA8M2Wmdc5ZhaggqtDvuXA70dwh99vXlLBBXRX1M7xZ368wTais4vPWBHkznpu0mUQqSdP2A1I+F2wScESaM3bqM5oHIMCpGaWfysd8lPm1S3BR6RWVTKhgRVaWa5NO6PCXwrEgau81EStYYKg3b6d16Dti0o4J75CoYH+Eov/bG571lvp7Yi8RiUdtLgZLu5nlUiYr6/nH8zZE9Eu5s+fhRZASO0pRAH4s2i4PfbgOuUc7FRySopSZiSJG4jsSG9IJLkXsETz4LKpkRN7YIX0iiA6ZsIOnBlcTj8W2XtiEcezO5oz5II7pXm1hxoqsy6RJEqnhTQkWhPDPO2gXhM4dqWir6BTBkgqac4vVhT3adO605CIMZ31K5+NZw3XHrWlDK8FVi4jJSYdDTGS7h17eSokvec6AdF4mHxoqn7x864lnizyJ/Ync24eXv5y8ev964LJ01xXmLOXirt8FDmxZ/4QT0fJw6HHpZ8d8CKYrViV0tqlWUD7LjAiYboZZHkfV5DRF7dF8efby9en79P3pyatBi91GmGMqRdLTL4Ul1JpSbcZWdqg6yVsmwYqCE2JWNLrCHNgUmKqN+VaT7ekcLJ6Qpo7oz+ZAaIztZ87pMycMF1bwt5fEOqTqhGpAqR1Glif+CsIN+7UJV3bplSScIDrluLDWzK6RLDBbNba4TVle8p6CuPH8OEqGVAH2psvN8lwa1VaIbA2kK4k9f3U6HI46BO61jLswdhtDjCbtxNAJsdIxygVBn/BHrVpvgd5FKG9XaF7IM0kU78RHSMAfDYfDo4uvVmQ+1Niv43BTeL7D9vF7xX5HE2sZ2E/x55nb0SorlQ2a7GE7eBcCcRvm7fdcwvx+Ojnes4f+SAtbl9GeA2zX7c4Rfl735oHyze/SJQvN9XLH/Vxhy2d5tgtZBn900AfxzuyRr6GgIUKiMrOGnZ7Itn+UtfFzJHmjlibi0zO67xXYqvp9f9pAkUAjmsVhwQtWgjYhOSShGTeCZ3bHJP31CnbLCgVFK3yKQPulbRhcMs+5XqmQf1PrYK09PGIg54+PLmC/eV8Y2LoqtWDzON5zoH3nYDb6StVgSnt2pSnO3tpNH2oaJsPb++yJbR98+Aqv82c9j9uc27bKX2XSHBPOO8qRplLgvX12Y8uCd4Nt7GF1nI11Ueq535fZtNkiyrU9ZcbR/IASiJmshd8ZjEMj4CbnzY2NmvafXHCLM4XXO86q3MuX/LtwCc8Z8H5F9oU+pIPVCjjBqyHFWXTXRu1APNzl3lw4e+ETsro1bPd2Re0AG6seU+oYdbHrp6JSNiqTsiyiLcTsNwLV9CSXwDU9cO0In6Dc6VL4WPNM+ITHWrc0pfyGNsqO/PYApTt2Z7d1FrSn68pndHq/28G0B4/Ct0SzKGl99x7egPe7B+5ECKHVniGH1XfWW635MgNWNtZnaB8Nh3g4ef/LKb0Nk0O8plOJPILs622Xr03gZky0sYltUUN6aCqyuSN18AiSmowG4mwgMsoFj20XDtCfPAr2zijsJ09kGGdQJF+Z41EC0ZoqkrTjaEgv+APrzfcOjvH8uD0wSzaXQbc7f1GcLKWms0mGj/farXTebtJdd9p0p3nnftjxPBYPxCH+Ix4kVe5WSErAWEGLeH706EDoTRg76AKRnF5G53U7cFqaCLjEgGgXeT5E3td9N1Tk7H0fXaBDli+PRwHpmo9plnfH7a4HMIfHB6Pk8UAs8mP+vTweHfaLFmdlIwty9DmHMcJDIMZeKWUP/SzQ6I/sYJ6r0pY1KM4w9lzSpVjjv3mdZyKqipU9XbBPh93zTO2LSa7jXmWD8CWoUdXLFlt2aKnTpkz1ahlV3RlF2AiKLIjzUh+P1JMB1U1g/e2zVnN6PsBLWGD/jfaVowoWqihjhGXRInfPD8RlTDLIQjItoALRAYQUjeRvQlcNUc65YhBFgPcMKhGLbwU9/4ju9myz50c7Jlu36+FCR2ShDIDS+RFk+IIQAFPt6+jCHx2crYoipe3FyHcETozsVqjqRoC+bIkjzDmgWkyh9LxZHF+6GUYd7UAAfIyqvmQOk8foGjFL1mTM+emaVjaBeicGjLAhoN/o/qtjlD2KQq1u65tOE2b2zg8cA1etBxRRTgLF8EynJHRirYCntiwqOlgLEwQsXEopJy0hpzCzdZlFtptVAu8D1z1t1hHp4mHc08cWgeBAnzXwyAZh9SN7jIooNIl7h/gUb+vycrrDtcQpmgaT5H3LRgIUsotwi9a0pPP8goSQlifpBcT9vaaN6Ac0B/HHrg2f3Jf+ZpF4JoYbUQsBX4PNUQsVP6OLuD8PJtiY737E8odqA4tD17XIplLOdqmugwPtePPBu/UbAcqXVzZI5xMydGvrlgLpcjaDPeF3fGITa4PqH/jECBs+aoADeDgQB/BJ8e3tOeYJslRnL1VDliO113ZMFB44s9f3jv35yEfsTYB/snH7bmA9EcUeUE17fA/Uce5u86h7sCHnBV4nrzlHs6c9N6Db60n+cKzueoP+IJotlzKkoGTaO2Ef7gZ1rZbUTBSSN4cLMO9xMOgVON6VbpWW+dlprvXFyOpUe9OCE5UNC3DUl3YiqgUYGgFLv8GXOOl492n8zZPt9kg+Y3d75C9l9uiLjJN69i+u2OsLa3d9gcIzrwE7wPnzn/1rEetd1yIslNvNs3+B1nAx1tHa0HIurxx1yVFrupHQbbXQaYQNx86C+ZgF86ETzN4JSTogaUO3/lrIk7tZ3Sag23SkD5lspO3SfElEQDHA3YckMWsQMkSjQwgE/nB4wGW3cxuW4M/F/xzGfIsj4zOYNlilzZJlCQ7e2HXc3hBd+rckwrCVs9ejrrLPMm+3Tjr75O2No1WfC3dJvyXz4cAnUq4P1w6IbRboQGx83EBzyyt+Bb5rW83f6V0tll+oW4/inRRY9xfEwuKmEy9evHvtVycie2H6SWyNAK+c8RjvDXYJBG0odW4ztqrp7grzNDaKJv8qs8wy/w9VlwcTSemqaVQl6HCwpY27nUoeMp3I2pw3F/E2wXk+2sk2abOD2OkX0Dr0jU8esQx0rvHTFH7sKfwZKym3AwpauOH5PhVMTO6KRDAQ/D8j7383hE2DunllxR4uYAsnQ/PIVIhDHs94/6HPWWfvHINpB5/ZW7pLmiJ67uIM9nP2epkrVQKRfe60PxDwsZtXDW2YGEkKonxcStf74hghEXLMh7x1tnCH8vYN7eLShYqgNEzlB751cCRmZGfYlNEdAJYyxCqfOPi98994L6IY0tkhe3dhZco8gwlt77dTRW8q4YRk4wV4vUNonYKlVP+jfd6vEdwRpad32LLvYXBHXLJ0muwn6oYf/qnh9tYfCL0eJbX6fZXX0D+S+m5U50m9V7Y33313ro+3btee0d9MNPpunD+210zXo/ae6faUO66ZBjDsXaY1lrg+5OHk41qVN06ZrQ9VHBOJufyo+P8aQtEpa+9cj0RWyysqJznZ58pMC2mb8fbGfMp+/cs8g4+Vv8KXPaHx7o6+kx1v+YNWEUk6SQXqu2IHl/Y5Bo93eLu929v/Bc87suU=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
